# Fairness Metrics and Group Diagnostics Lab


## Purpose

This lab introduces group fairness metrics.

Learning goals:

- Build a synthetic classification dataset.
- Train a classifier.
- Compute selection rates, true positive rates, false positive rates, and predictive values.
- Interpret fairness gaps as governance signals.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)

X, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    weights=[0.65, 0.35],
    random_state=42,
)

group = rng.choice(["A", "B"], size=len(y), p=[0.55, 0.45])

X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(
    X,
    y,
    group,
    test_size=0.30,
    stratify=y,
    random_state=42,
)

model = Pipeline([
    ("scale", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)

score = model.predict_proba(X_test)[:, 1]
prediction = (score >= 0.5).astype(int)

audit = pd.DataFrame({
    "target": y_test,
    "group": group_test,
    "score": score,
    "prediction": prediction,
})

rows = []

for group_name, data in audit.groupby("group"):
    tn, fp, fn, tp = confusion_matrix(data["target"], data["prediction"], labels=[0, 1]).ravel()

    rows.append({
        "group": group_name,
        "selection_rate": data["prediction"].mean(),
        "base_rate": data["target"].mean(),
        "true_positive_rate": tp / max(tp + fn, 1),
        "false_positive_rate": fp / max(fp + tn, 1),
        "positive_predictive_value": tp / max(tp + fp, 1),
    })

metrics = pd.DataFrame(rows)
metrics

## Interpretation

Group metrics are not final moral answers. They are evidence used in governance review.